# Gas Physisorption: BET Surface Area & BJH Pore Distribution

This notebook demonstrates quantitative gas sorption analysis:
- $\text{N}_2$ 77.35 K adsorption-desorption isotherm parsing & branch separation
- Multi-point linear BET regression transform:
  $$\frac{p/p_0}{q(1 - p/p_0)} = \frac{1}{q_m C} + \frac{C-1}{q_m C} \left(\frac{p}{p_0}\right)$$
- Rigorous IUPAC Rouquerol consistency criteria verification
- BJH mesopore size distribution ($dV/dD$) via Halsey thickness and Kelvin capillary condensation equations
- Multi-technique cross-correlation: BET specific surface area vs double-layer capacitance ($C_{\text{dl}}$).

In [1]:
import sys
from pathlib import Path
root_path = Path.cwd().parent.parent if Path.cwd().name == 'characterization' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import matplotlib.pyplot as plt
import numpy as np
from parsers.characterization import parse_isotherm_file
from modules.characterization.bet import (
    calculate_bet_surface_area,
    calculate_bjh_pore_distribution,
    plot_isotherm,
    plot_bet_linear,
    plot_bjh_pore_distribution,
)
from modules.characterization.comparative import (
    correlate_cross_technique_metrics,
    plot_cross_technique_correlation,
)
from scripts.utils import apply_publication_style

apply_publication_style()
print('Physisorption modules loaded.')

Physisorption modules loaded.


## 1. Type IV Isotherm & BET Surface Area with Rouquerol Checks
Inspect $\text{N}_2$ isotherm with H1 hysteresis loop.

In [2]:
iso_path = root_path / 'data' / 'examples' / 'characterization' / 'bet' / 'mesoporous_catalyst_n2_isotherm.csv'
iso_data = parse_isotherm_file(iso_path)
df_ads = iso_data.dataframe[iso_data.dataframe['branch'] == 'adsorption']
df_des = iso_data.dataframe[iso_data.dataframe['branch'] == 'desorption']

fig, ax = plot_isotherm(iso_data, title='Type IV N2 Adsorption-Desorption Isotherm (77.3 K)')
plt.show()

bet_res = calculate_bet_surface_area(df_ads['p_p0'].values, df_ads['quantity_adsorbed'].values, p_range=(0.05, 0.30))
print(f'S_BET: {bet_res["s_bet_m2_g"]:.1f} m^2/g, C constant = {bet_res["c_constant"]:.1f}, R^2 = {bet_res["r_squared"]:.5f}')
rouq = bet_res['rouquerol_criteria']
print(f'Rouquerol Check: Monotonic={rouq["monotonic_increasing"]}, C>0={rouq["c_positive"]}, Overall Valid={rouq["overall_valid"]}')

fig2, ax2 = plot_bet_linear(bet_res, title='BET Linear Transform')
plt.show()

S_BET: 152.6 m^2/g, C constant = 112.9, R^2 = 1.00000
Rouquerol Check: Monotonic=True, C>0=True, Overall Valid=True


C:\Users\mjram\python_graphing\modules\characterization\bet.py:254: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_1632\1841884514.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mjram\python_graphing\modules\characterization\bet.py:311: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_1632\1841884514.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. BJH Mesopore Size Distribution (dV/dD)
Calculate incremental and differential pore size distribution on desorption branch.

In [3]:
bjh_res = calculate_bjh_pore_distribution(df_des['p_p0'].values, df_des['quantity_adsorbed'].values, branch='desorption')
print(f'Total Mesopore Volume: {bjh_res["total_pore_volume_cm3_g"]:.3f} cm^3/g')
print(f'Mean Pore Diameter: {bjh_res["mean_pore_diameter_nm"]:.1f} nm')

fig, ax = plot_bjh_pore_distribution(bjh_res, title='BJH Mesopore Size Distribution (Desorption Branch)')
plt.show()

Total Mesopore Volume: 0.150 cm^3/g
Mean Pore Diameter: 6.2 nm


C:\Users\mjram\python_graphing\modules\characterization\bet.py:367: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_1632\73495312.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Multi-Technique Correlation: BET Surface Area vs Double-Layer Capacitance
Correlate gas physisorption physical area with electrochemical active surface metrics across 5 catalyst synthesis batches.

In [4]:
bet_areas = [45.0, 92.0, 148.0, 210.0, 280.0]
c_dl = [12.5, 23.8, 38.2, 53.0, 71.4]
corr = correlate_cross_technique_metrics(
    x_values=bet_areas,
    y_values=c_dl,
    x_label='BET Specific Surface Area $S_{\\mathrm{BET}}~[\\mathrm{m^2\\cdot g^{-1}}]$',
    y_label='Double-Layer Capacitance $C_{\\mathrm{dl}}~[\\mathrm{mF\\cdot cm^{-2}}]$',
    sample_labels=['Cat-1', 'Cat-2', 'Cat-3', 'Cat-4', 'Cat-5'],
)
print(f'Correlation R^2: {corr["r_squared"]:.4f}, p-value: {corr["p_value"]:.3e}')
print(f'Disclaimer: {corr["disclaimer"]}')

fig, ax = plot_cross_technique_correlation(corr, title='Cross-Technique Validation: BET Area vs Double-Layer Capacitance')
plt.show()

Correlation R^2: 0.9997, p-value: 1.690e-06
Disclaimer: Correlations between physical properties and catalytic performance do not establish direct causality without independent mechanistic verification.


C:\Users\mjram\python_graphing\modules\characterization\comparative.py:164: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_1632\140362491.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
